#QUES 1

In [1]:
import pandas as pd

roll_number = input("Enter your college roll number: ").strip()

digits = ''.join(ch for ch in roll_number if ch.isdigit())

if len(digits) < 2:
    raise ValueError("Roll number must contain at least two digits.")

last_two = digits[-2:]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

categories = ["billing", "account", "general"]

personalized_entries = []

for digit in last_two:
    category = categories[int(digit) % 3]

    if category == "billing":
        question = "how do I update my billing details"
        answer = "Go to Billing > Payment Details to update your billing information."
        keywords = "billing fee payment details update"

    elif category == "account":
        question = "how do I update my account details"
        answer = "Go to Account > Profile to update your account details."
        keywords = "account profile update details"

    else:
        question = "how do I update my general details"
        answer = "Go to Help > General Information to update your general details."
        keywords = "general information update details"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

df = pd.DataFrame(fixed_entries + personalized_entries)

display(df)

Enter your college roll number:  1024170006


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing
5,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing


#QUES 2

In [2]:
def score_hypothesis(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        matched_words = query_words & (question_words | keyword_words)

        if matched_words:
            score = len(matched_words) / len(query_words)

            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "matched_words": ", ".join(sorted(matched_words))
            })

    if len(results) == 0:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values(
        "score", ascending=False
    ).reset_index(drop=True)


query = input("Enter your query: ")

result = score_hypothesis(query, df)

display(result)

Enter your query:  fee payment


,question,answer,category,score,matched_words
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1.0,"fee, payment"
1,how do I update my billing details,Go to Billing > Payment Details to update your...,billing,1.0,"fee, payment"
2,how do I update my billing details,Go to Billing > Payment Details to update your...,billing,1.0,"fee, payment"
3,what is the annual fee,The annual fee is Rs 500.,billing,0.5,fee


#QUES 3

In [3]:
def same_category(category_name, df):
    return df[
        df["category"].str.lower() == category_name.lower()
    ].reset_index(drop=True)


category_name = input("Enter category: ")

result = same_category(category_name, df)

display(result)

Enter category:  billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
2,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing
3,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing


#QUES 4

In [4]:
print("FAQ Entries:")

for i, question in enumerate(df["question"]):
    print(i, "->", question)

entry_index = int(input("Enter entry index (0-5): "))
new_keyword = input("Enter new keyword: ").strip().lower()

existing_keywords = df.loc[entry_index, "keywords"].split()

if new_keyword not in existing_keywords:
    existing_keywords.append(new_keyword)

df.loc[entry_index, "keywords"] = " ".join(existing_keywords)

csv_filename = roll_number + "_faq_data.csv"

df.to_csv(csv_filename, index=False)

print("\nUpdated DataFrame:")
display(df)

print("CSV saved as:", csv_filename)

FAQ Entries:
0 -> what is the annual fee
1 -> how to reset password
2 -> what are your working hours
3 -> how can i pay the fee
4 -> how do I update my billing details
5 -> how do I update my billing details


Enter entry index (0-5):  1
Enter new keyword:  charges 



Updated DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login charges,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing
5,how do I update my billing details,Go to Billing > Payment Details to update your...,billing fee payment details update,billing


CSV saved as: 1024170006_faq_data.csv


#QUES 5

In [5]:
category_count = df.groupby("category").size().reset_index(name="count")

display(category_count)

,category,count
0,account,1
1,billing,4
2,general,1


#QUES 6

In [6]:
def score_hypothesis_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():

        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        matched_words = query_words & (question_words | keyword_words)

        if matched_words:
            score = len(matched_words) / len(query_words)

            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "matched_words": ", ".join(sorted(matched_words))
            })

    if len(results) == 0:
        return pd.DataFrame()

    result_df = pd.DataFrame(results)

    max_score = result_df["score"].max()

    return result_df[
        result_df["score"] == max_score
    ].reset_index(drop=True)

In [7]:
query = "fee"

result = score_hypothesis_with_ties(query, df)

print("Query:", query)
display(result)

Query: fee


,question,answer,category,score,matched_words
0,what is the annual fee,The annual fee is Rs 500.,billing,1.0,fee
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1.0,fee
2,how do I update my billing details,Go to Billing > Payment Details to update your...,billing,1.0,fee
3,how do I update my billing details,Go to Billing > Payment Details to update your...,billing,1.0,fee


In [8]:
query = "password"

result = score_hypothesis_with_ties(query, df)

print("Query:", query)
display(result)

Query: password


,question,answer,category,score,matched_words
0,how to reset password,Go to Settings > Reset Password.,account,1.0,password
